In [33]:
from torch.utils.data import Dataset
import pandas as pd
import os
from PIL import Image
from torchvision.transforms import Resize, ToTensor, Compose
import torchvision
from torch import nn
from torch.nn import Module
import torch
import torch.distributed as dist
from torch.optim.optimizer import Optimizer, required
import re
import argparse
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter
import os

# 1. Dataset

In [34]:
class TransformsSimCLR:
    """
    A stochastic data augmentation module that transforms any given data example randomly
    resulting in two correlated views of the same example,
    denoted x ̃i and x ̃j, which we consider as a positive pair.
    """

    def __init__(self, size):
        s = 1
        color_jitter = torchvision.transforms.ColorJitter(
            0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s
        )
        self.train_transform = torchvision.transforms.Compose(
            [
                torchvision.transforms.RandomResizedCrop(size=size),
                torchvision.transforms.RandomHorizontalFlip(),  # with 0.5 probability
                torchvision.transforms.RandomApply([color_jitter], p=0.8),
                torchvision.transforms.RandomGrayscale(p=0.2),
                torchvision.transforms.ToTensor(),
            ]
        )


    def __call__(self, x):
        return self.train_transform(x), self.train_transform(x)

In [35]:
simclr_data_transform = {
    "training": TransformsSimCLR((450,200)),
    "test": Compose([
        Resize(size=(450,200)),
        ToTensor()
    ])
}

class MammoDataset(Dataset):
    def __init__(self, annotation_path: str = "/mnt/d/Research/data/Mammo/split_data.csv",
                 image_folder_path: str = "/mnt/d/Research/data/Mammo/Processed_Images_450_200",
                 phase: str = "training",
                 transform: object = None):
        df = pd.read_csv(annotation_path)
        self.data = df.loc[df['split'] == phase].reset_index()

        self.phase = phase
        self.data_path = image_folder_path
        self.transform = simclr_data_transform[self.phase] if transform == None else transform

    
    def get_score(self, index):
        birads = self.data['breast_birads'].iloc[index]
        score = eval(birads[-1])
        return score
    
    def get_path(self, index):
        image_name = self.data['image_id'].iloc[index]
        study_id = self.data['study_id'].iloc[index]
        image_path = os.path.join(self.data_path, study_id + "/" + image_name + ".png")
        return image_path
    
    def __len__(self):
        return len(self.data.index)
    
    def __getitem__(self, index):
        image_path = self.get_path(index=index)
        image = Image.open(image_path)
        if self.transform:
            image = self.transform(image)
        
        label = self.get_score(index=index) - 1
        return image, label

# 2. Model

In [36]:
def get_resnet(name, pretrained=False):
    resnets = {
        "resnet18": torchvision.models.resnet18(weights=pretrained),
        "resnet50": torchvision.models.resnet50(weights=pretrained),
    }
    if name not in resnets.keys():
        raise KeyError(f"{name} is not a valid ResNet version")
    return resnets[name]

In [37]:
def get_encoder(model_name, output_dim: int = 256):
    encoder = get_resnet(model_name)
    encoder.fc =  nn.Sequential(nn.Linear(encoder.fc.in_features, 1000),
                                nn.ReLU(),
                                nn.Dropout(0.1),
                                nn.Linear(1000, output_dim))
    
    return encoder

In [38]:
class Encoder(Module):
    def __init__(self, model_name) -> None:
        super(Encoder, self).__init__()
        self.encoder = get_encoder(model_name)

    def forward_once(self, x):
        return self.encoder(x)
    
    def forward(self, input1, input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        return output1, output2

# 3.Loss

In [39]:
class GatherLayer(torch.autograd.Function):
    """Gather tensors from all process, supporting backward propagation."""

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        output = [torch.zeros_like(input) for _ in range(dist.get_world_size())]
        dist.all_gather(output, input)
        return tuple(output)

    @staticmethod
    def backward(ctx, *grads):
        (input,) = ctx.saved_tensors
        grad_out = torch.zeros_like(input)
        grad_out[:] = grads[dist.get_rank()]
        return grad_out

In [40]:
class NT_Xent(nn.Module):
    def __init__(self, batch_size, temperature, world_size):
        super(NT_Xent, self).__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.world_size = world_size

        self.mask = self.mask_correlated_samples(batch_size, world_size)
        self.criterion = nn.CrossEntropyLoss(reduction="sum")
        self.similarity_f = nn.CosineSimilarity(dim=2)

    def mask_correlated_samples(self, batch_size, world_size):
        N = 2 * batch_size * world_size
        mask = torch.ones((N, N), dtype=bool)
        mask = mask.fill_diagonal_(0)
        for i in range(batch_size * world_size):
            mask[i, batch_size * world_size + i] = 0
            mask[batch_size * world_size + i, i] = 0
        return mask

    def forward(self, z_i, z_j):
        """
        We do not sample negative examples explicitly.
        Instead, given a positive pair, similar to (Chen et al., 2017), we treat the other 2(N − 1) augmented examples within a minibatch as negative examples.
        """
        N = 2 * self.batch_size * self.world_size

        if self.world_size > 1:
            z_i = torch.cat(GatherLayer.apply(z_i), dim=0)
            z_j = torch.cat(GatherLayer.apply(z_j), dim=0)
        z = torch.cat((z_i, z_j), dim=0)

        sim = self.similarity_f(z.unsqueeze(1), z.unsqueeze(0)) / self.temperature

        sim_i_j = torch.diag(sim, self.batch_size * self.world_size)
        sim_j_i = torch.diag(sim, -self.batch_size * self.world_size)

        # We have 2N samples, but with Distributed training every GPU gets N examples too, resulting in: 2xNxN
        positive_samples = torch.cat((sim_i_j, sim_j_i), dim=0).reshape(N, 1)
        negative_samples = sim[self.mask].reshape(N, -1)

        labels = torch.zeros(N).to(positive_samples.device).long()
        logits = torch.cat((positive_samples, negative_samples), dim=1)
        loss = self.criterion(logits, labels)
        loss /= N
        return loss

# 4. Optimization

In [41]:
EETA_DEFAULT = 0.001


class LARS(Optimizer):
    """
    Layer-wise Adaptive Rate Scaling for large batch training.
    Introduced by "Large Batch Training of Convolutional Networks" by Y. You,
    I. Gitman, and B. Ginsburg. (https://arxiv.org/abs/1708.03888)
    """

    def __init__(
        self,
        params,
        lr=required,
        momentum=0.9,
        use_nesterov=False,
        weight_decay=0.0,
        exclude_from_weight_decay=None,
        exclude_from_layer_adaptation=None,
        classic_momentum=True,
        eeta=EETA_DEFAULT,
    ):
        """Constructs a LARSOptimizer.
        Args:
        lr: A `float` for learning rate.
        momentum: A `float` for momentum.
        use_nesterov: A 'Boolean' for whether to use nesterov momentum.
        weight_decay: A `float` for weight decay.
        exclude_from_weight_decay: A list of `string` for variable screening, if
            any of the string appears in a variable's name, the variable will be
            excluded for computing weight decay. For example, one could specify
            the list like ['batch_normalization', 'bias'] to exclude BN and bias
            from weight decay.
        exclude_from_layer_adaptation: Similar to exclude_from_weight_decay, but
            for layer adaptation. If it is None, it will be defaulted the same as
            exclude_from_weight_decay.
        classic_momentum: A `boolean` for whether to use classic (or popular)
            momentum. The learning rate is applied during momeuntum update in
            classic momentum, but after momentum for popular momentum.
        eeta: A `float` for scaling of learning rate when computing trust ratio.
        name: The name for the scope.
        """

        self.epoch = 0
        defaults = dict(
            lr=lr,
            momentum=momentum,
            use_nesterov=use_nesterov,
            weight_decay=weight_decay,
            exclude_from_weight_decay=exclude_from_weight_decay,
            exclude_from_layer_adaptation=exclude_from_layer_adaptation,
            classic_momentum=classic_momentum,
            eeta=eeta,
        )

        super(LARS, self).__init__(params, defaults)
        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay
        self.use_nesterov = use_nesterov
        self.classic_momentum = classic_momentum
        self.eeta = eeta
        self.exclude_from_weight_decay = exclude_from_weight_decay
        # exclude_from_layer_adaptation is set to exclude_from_weight_decay if the
        # arg is None.
        if exclude_from_layer_adaptation:
            self.exclude_from_layer_adaptation = exclude_from_layer_adaptation
        else:
            self.exclude_from_layer_adaptation = exclude_from_weight_decay

    def step(self, epoch=None, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        if epoch is None:
            epoch = self.epoch
            self.epoch += 1

        for group in self.param_groups:
            weight_decay = group["weight_decay"]
            momentum = group["momentum"]
            eeta = group["eeta"]
            lr = group["lr"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                param = p.data
                grad = p.grad.data

                param_state = self.state[p]

                # TODO: get param names
                # if self._use_weight_decay(param_name):
                grad += self.weight_decay * param

                if self.classic_momentum:
                    trust_ratio = 1.0

                    # TODO: get param names
                    # if self._do_layer_adaptation(param_name):
                    w_norm = torch.norm(param)
                    g_norm = torch.norm(grad)

                    device = g_norm.get_device()
                    trust_ratio = torch.where(
                        w_norm.ge(0),
                        torch.where(
                            g_norm.ge(0),
                            (self.eeta * w_norm / g_norm),
                            torch.Tensor([1.0]).to(device),
                        ),
                        torch.Tensor([1.0]).to(device),
                    ).item()

                    scaled_lr = lr * trust_ratio
                    if "momentum_buffer" not in param_state:
                        next_v = param_state["momentum_buffer"] = torch.zeros_like(
                            p.data
                        )
                    else:
                        next_v = param_state["momentum_buffer"]

                    next_v.mul_(momentum).add_(scaled_lr, grad)
                    if self.use_nesterov:
                        update = (self.momentum * next_v) + (scaled_lr * grad)
                    else:
                        update = next_v

                    p.data.add_(-update)
                else:
                    raise NotImplementedError

        return loss

    def _use_weight_decay(self, param_name):
        """Whether to use L2 weight decay for `param_name`."""
        if not self.weight_decay:
            return False
        if self.exclude_from_weight_decay:
            for r in self.exclude_from_weight_decay:
                if re.search(r, param_name) is not None:
                    return False
        return True

    def _do_layer_adaptation(self, param_name):
        """Whether to do layer-wise learning rate adaptation for `param_name`."""
        if self.exclude_from_layer_adaptation:
            for r in self.exclude_from_layer_adaptation:
                if re.search(r, param_name) is not None:
                    return False
        return True

# 5. Pipeline

In [42]:
def train(train_loader, model, criterion, optimizer, writer, device):
    loss_epoch = 0
    model = model.to(device)
    for step, ((x_i, x_j), _) in enumerate(train_loader):
        optimizer.zero_grad()
        x_i = x_i.to(device)
        x_j = x_j.to(device)

        # positive pair, with encoding
        z_i, z_j = model(x_i, x_j)

        loss = criterion(z_i, z_j)
        loss.backward()

        optimizer.step()


        if nr == 0 and step % 50 == 0:
            print(f"Step [{step}/{len(train_loader)}]\t Loss: {loss.item()}")

        if nr == 0:
            writer.add_scalar("Loss/train_epoch", loss.item(), global_step)
            global_step += 1
        loss_epoch += loss.item()
    return loss_epoch


def save_model(model):
    out = os.path.join(model_path, "checkpoint_{}.tar".format(current_epoch))

    # To save a DataParallel model generically, save the model.module.state_dict().
    # This way, you have the flexibility to load the model any way you want to any device you want.
    if isinstance(model, torch.nn.DataParallel):
        torch.save(model.module.state_dict(), out)
    else:
        torch.save(model.state_dict(), out)

In [43]:
phase = "training"
batch_size =8
encoder ="resnet50"
temperature = 0.5
optimizer="Adam"
lr = 3e-4
weight_decay =1e-9 
nr = 0
epochs = 100
model_path = "/kaggle/working/"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
train_dataset = MammoDataset(
    annotation_path="/kaggle/input/mammo-224-224-ver2/mammo_224_224/split_data.csv",
    image_folder_path="/kaggle/input/mammo-450-200-ver4/Processed_Images_450_200",
    phase=phase
)

trainloader = DataLoader(train_dataset, 
                         batch_size=batch_size, 
                         shuffle=True,
                         drop_last=True)
encoder = Encoder(encoder)
# encoder.load_state_dict(torch.load("/mnt/d/Research/pretrain/checkpoint_100.tar"), strict=False)
criterion = NT_Xent(batch_size, temperature, 1)
if optimizer == "Adam":
    optimizer = AdamW(encoder.parameters(), lr=lr)
if optimizer == "LARS":
    learning_rate = 0.3 * batch_size / 256
    optimizer = LARS(
        encoder.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
        exclude_from_weight_decay=["batch_normalization", "bias"],
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, epochs, eta_min=0, last_epoch=-1
    )
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if nr == 0:
    writer = SummaryWriter()

global_step = 0
current_epoch = 0
for epoch in range(epochs):
    lr = optimizer.param_groups[0]["lr"]

    loss_epoch = train(
        train_loader=trainloader,
        model = encoder, 
        criterion = criterion,
        optimizer = optimizer,
        writer = writer,
        device = device)

    if nr == 0 and scheduler:
        scheduler.step()

    if nr == 0 and epoch % 10 == 0:
        save_model( encoder)

    if nr == 0:
        writer.add_scalar("Loss/train", loss_epoch / len(trainloader), epoch)
        writer.add_scalar("Misc/learning_rate", lr, epoch)
        print(
            f"Epoch [{epoch}/{epochs}]\t Loss: {loss_epoch / len(trainloader)}\t lr: {round(lr, 5)}"
        )
        current_epoch += 1

## end training
save_model(encoder)

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Step [0/1600]	 Loss: 2.7408695220947266


UnboundLocalError: local variable 'global_step' referenced before assignment